# COMP5339 Assignment 2

GROUP: TUT11-ASSIGNMENTGRP-10 <br>

SID <br>
- 540969766
- 540931475

In [18]:
import pandas as pd
import numpy as np
import requests
import time
from datetime import datetime, timedelta
from dotenv import load_dotenv
import os
import ast


## Data Retrieval

In [2]:
# Load environment variables
load_dotenv()
API_KEY = os.getenv("API_KEY")

# Error handling for API key
if not API_KEY:
    raise ValueError("API_KEY is not set in the environment variable. Check .env file.")

base_url = "https://api.openelectricity.org.au/v4"

# Get the data from the API
def get_facilities():
    url = f'{base_url}/facilities/'
    params = {
        'interval': '5m',
        'network_id': 'NEM'
    }

    # Authorisation
    headers = {'Authorization': f'Bearer {API_KEY}'}

    # Make the request
    response = requests.get(url, headers = headers, params = params)
    facilities = pd.DataFrame(response.json())
    facilities.to_csv('facilities.csv', index = False)
    time.sleep(60) # wait for 60 seconds before the next request

    return facilities
    

In [3]:
# Get the metadata from the API
# get_facilities()
facilities = pd.read_csv('facilities.csv')
facilities.head()


,version,created_at,success,data,total_records
0,4.3.0,2025-10-27T15:08:28+11:00,True,"{'code': 'ADP', 'name': 'Adelaide Desalination...",514
1,4.3.0,2025-10-27T15:08:28+11:00,True,"{'code': 'ALDGASF', 'name': 'Aldoga', 'network...",514
2,4.3.0,2025-10-27T15:08:28+11:00,True,"{'code': 'AMCORGR', 'name': 'Amcor Glass', 'ne...",514
3,4.3.0,2025-10-27T15:08:28+11:00,True,"{'code': 'ANGASTON', 'name': 'Angaston', 'netw...",514
4,4.3.0,2025-10-27T15:08:28+11:00,True,"{'code': 'APS', 'name': 'Anglesea', 'network_i...",514


### Preprocessing for metadata

get_facilities 
일단 먼저 facilities 데이터를 가져와서
- facility_code 를 기준으로 name, location을 추출
  - 그리고 그 아래 unit_id 를 기준으로 code, fueltech_id, status_id, capacity_registered, capacity_maximum, dispatch_type를 정리
- unique facility_code를 list로 저장

get_facility_power
- 위에서 저장한 facility_code list 기준으로 원하는 기간의 power/energy 데이터 가져오기
- 그래서 새로운 table에는 facility_code, facility_name, location

In [29]:
class Dataloader:
    '''
    Handle data loading and preprocessing
    '''
    def __init__(self):
        '''
        Initialize the Dataloader with the API key  
        '''
        load_dotenv()
        api_key = os.getenv("API_KEY")

        if not api_key:
            raise ValueError("API_KEY is not set in the environment variable. Check .env file.")
        
        self.api_key = api_key
        self.base_url = "https://api.openelectricity.org.au/v4"
        self.headers = {'Authorization': f'Bearer {api_key}'}

        print("Ready to load data")
    
    def save_to_csv(self, df, filename):
        '''
        Save the dataframe to a csv file
        '''
        if not df.empty:
            df.to_csv(filename, index = False)
            print(f'Successfully saved data to {filename}')
        else:
            print("The dataframe is empty. No data to save.")


    def get_facilities(self):
        ''' 
        Get the facilities data from the API
        '''
        facilities_url = f'{self.base_url}/facilities/'
        params = {
            'interval': '5m',
            'network_id': 'NEM'
        }

        try:
            response = requests.get(facilities_url, headers = self.headers, params = params)
            facilities = pd.DataFrame(response.json())

            print(f'Successfully retrieved data for {len(facilities)} facilities')

            self.save_to_csv(facilities, 'facilities_raw.csv')
            
            time.sleep(60) # wait for 60 seconds before the next request

            return facilities
        
        except requests.exceptions.RequestException as e:
            print(f"Error occurred: {e}")
            return None
        

    def cleaning_facilities(self, facilities):
        '''
        Clean the facilities data
        '''
        cleaned_facilities = []

        for idx, row in facilities.iterrows():

            try:
                facility_data = ast.literal_eval(row['data']) # string to dict

                # Extract the basic information first
                facility_info = {
                    'facility_code': facility_data.get('code'),
                    'facility_name': facility_data.get('name'),
                    'facility_region': facility_data.get('network_region'),
                    'units': []
                }

                # Extract the location
                location = facility_data.get('location', {})
                facility_info['latitude'] = location.get('lat')
                facility_info['longitude'] = location.get('lng')

                # Get the unit details
                units = facility_data.get('units', [])


                for unit in units:
                    if unit.get('status_id') == 'operating':
                        unit_info = {
                            'unit_code': unit.get('code'),
                            'fuel_type': unit.get('fueltech_id'),
                            'capacity_registered': unit.get('capacity_registered'),
                            'capacity_maximum': unit.get('capacity_maximum'),
                            'emissions_co2': unit.get('emissions_factor_co2')
                        }

                        facility_info['units'].append(unit_info)
                        
            
                cleaned_facilities.append(facility_info)
            
            except Exception as e:
                print(f"Error processing facility {idx}: {e}")
                continue


        return pd.DataFrame(cleaned_facilities)
            


        
        

    # def get_facility_details(self, facilities, start_date, end_date, metric):
    #     '''
    #     Get the facility power generation data
    #     '''
    #     power_url = f'{self.base_url}/data/facilities/NEM'
    #     params = {
    #         'metrics': metric,
    #         'interval': '5m',
    #         'date_start': start_date,
    #         'date_end': end_date
    #     }

    #     facility_codes = facilities['facility_code'].unique()

    #     for facility_code in facility_codes:
    #         params['facility_code'] = facility_code
    #         response = requests.get(power_url, headers = self.headers, params = params)
    #         facility_data = response.json()

            
            



In [4]:
loader = Dataloader()

facilities = loader.get_facilities()
facilities.head()

Ready to load data
Successfully retrieved data for 514 facilities
Successfully saved data to facilities_raw.csv


,version,created_at,success,data,total_records
0,4.3.0,2025-10-28T15:42:27+11:00,True,"{'code': 'ADP', 'name': 'Adelaide Desalination...",514
1,4.3.0,2025-10-28T15:42:27+11:00,True,"{'code': 'ALDGASF', 'name': 'Aldoga', 'network...",514
2,4.3.0,2025-10-28T15:42:27+11:00,True,"{'code': 'AMCORGR', 'name': 'Amcor Glass', 'ne...",514
3,4.3.0,2025-10-28T15:42:27+11:00,True,"{'code': 'ANGASTON', 'name': 'Angaston', 'netw...",514
4,4.3.0,2025-10-28T15:42:27+11:00,True,"{'code': 'APS', 'name': 'Anglesea', 'network_i...",514


In [30]:
facilities = pd.read_csv('facilities_raw.csv')

loader = Dataloader()

samples = facilities[0:10]

fac = loader.cleaning_facilities(samples)
fac


Ready to load data


,facility_code,facility_name,facility_region,units,latitude,longitude
0,ADP,Adelaide Desalination,SA1,"[{'unit_code': 'ADPPV1', 'fuel_type': 'solar_u...",-35.096948,138.484061
1,ALDGASF,Aldoga,QLD1,"[{'unit_code': 'ALDGASF1', 'fuel_type': 'solar...",-23.839544,151.084900
2,AMCORGR,Amcor Glass,SA1,[],-34.882663,138.577975
3,ANGASTON,Angaston,SA1,"[{'unit_code': 'ANGAST1', 'fuel_type': 'distil...",-34.503948,139.024296
4,APS,Anglesea,VIC1,[],-38.389031,144.180589
5,APPIN,Appin,NSW1,"[{'unit_code': 'APPIN', 'fuel_type': 'gas_wcmg...",-34.210868,150.792711
6,ARWF,Ararat,VIC1,"[{'unit_code': 'ARWF1', 'fuel_type': 'wind', '...",-37.263393,143.082116
7,AVLSF,Avonlie,NSW1,"[{'unit_code': 'AVLSF1', 'fuel_type': 'solar_u...",-34.919115,146.609540
8,AWABAREF,Awaba,NSW1,[],-33.023339,151.551228
9,DEIBDL,Bairnsdale,VIC1,"[{'unit_code': 'BDL01', 'fuel_type': 'gas_ocgt...",-37.842231,147.563725


## Data Integration and Materialisation/Cashing

- whether the per-facility power generated is power or energy
  - power: Instantaneous power output/consumption (MW)
  - energy: Energy generated/consumed over time (MWh)

- per-market price and demand -> is this about price and demand per region (NSW, SA) or per fuel type

- for visualisation:
  - is it okay to show powerstation name and current power output or emissions or should we have to keep those information floating
  - the latest power production and emissions data meaning the overall data for that certain 